In [ ]:
import os
# change working directory so that it picks up the grapehelper library
os.chdir('/home/ftorgano/rna-kg-analysis')
print(os.getcwd())

In [ ]:
from helper_lib import graph
from helper_lib import cache
from helper_lib import predict
import importlib
importlib.reload(graph)
importlib.reload(cache)
importlib.reload(predict)
cache.set_embedding_cache_dir("./RNA-KG_notebooks/Default_RNA-KG/cache/embeddings/")
import logging
logging.basicConfig(level=logging.INFO)
logging.getLogger().setLevel(logging.INFO)

In [ ]:
from grape import GraphVisualizer
from sklearn.metrics import balanced_accuracy_score
import pandas as pd
from itables import init_notebook_mode,show
init_notebook_mode(all_interactive=False)
import time
import numpy as np
import matplotlib.pyplot as plt
cycler_colors = ["#3f90da", "#ffa90e", "#bd1f01", "#94a4a2", "#832db6", "#a96b59", "#e76300", "#b9ac70", "#717581", "#92dadd"]
from cycler import cycler
plt.rcParams['axes.prop_cycle'] = cycler(color=cycler_colors)
df_formatters = {'balanced_acc_mean':'{:.2%}'.format,'balanced_acc_std':'{:.2%}'.format}

In [ ]:
undirected_rnakg = graph.load_rnakg_fixed()

In [ ]:
from grape.edge_prediction import DecisionTreeEdgePrediction,RandomForestEdgePrediction
tree = DecisionTreeEdgePrediction()
train_sets = []
test_sets = []
for i in range(5):
    train,test = tree.split_graph_following_evaluation_schema(
        undirected_rnakg, 
        evaluation_schema="Connected Monte Carlo",
        number_of_holdouts=5, 
        holdout_number=i,
        train_size = 0.7,
        random_state=42
    )
    train_sets.append(train)
    test_sets.append(test)

In [ ]:
for i in range(5):
    print(f"Train {i}: {train_sets[i].get_number_of_edges()} edges, Test {i}: {test_sets[i].get_number_of_edges()} edges")

# 10D embedding

In [ ]:
# load embedding
embedding_bfs_10 = cache.load_embeddings('Node2VecSkipGramEnsmallen_BFS_10_fixed')
print(type(embedding_bfs_10))

## Parameter Tuning on Decision Tree

In [ ]:
from grape.edge_prediction import DecisionTreeEdgePrediction
from grape.edge_prediction import edge_prediction_evaluation

evaluation_schema = "Connected Monte Carlo"
edge_embedding_method = "Concatenate"
use_scale_free_distribution = True
training_unbalance_rate = 1/1
validation_unbalance_rate = 1/1

tree_10_tuning_results = []

for (i,train_graph) in enumerate(train_sets):
    results_10_tuning = pd.concat([
        edge_prediction_evaluation(
            graphs=train_graph,

            evaluation_schema = evaluation_schema,
            number_of_holdouts = 1,
            holdouts_kwargs=dict(train_size=0.7),
            use_scale_free_distribution=use_scale_free_distribution,
            validation_unbalance_rates=validation_unbalance_rate,
            
            models=[
                DecisionTreeEdgePrediction(
                    edge_embedding_methods = edge_embedding_method,
                    use_scale_free_distribution = use_scale_free_distribution,
                    training_unbalance_rate = training_unbalance_rate,
                ),
                DecisionTreeEdgePrediction(
                    edge_embedding_methods = edge_embedding_method,
                    use_scale_free_distribution = use_scale_free_distribution,
                    training_unbalance_rate = training_unbalance_rate,
                    max_features = 5,
                    max_depth = 50,
                ),
                DecisionTreeEdgePrediction(
                    edge_embedding_methods = edge_embedding_method,
                    use_scale_free_distribution = use_scale_free_distribution,
                    training_unbalance_rate = training_unbalance_rate,
                    max_features='sqrt',
                    max_depth = 200
                ),
                DecisionTreeEdgePrediction(
                    edge_embedding_methods = edge_embedding_method,
                    use_scale_free_distribution = use_scale_free_distribution,
                    training_unbalance_rate = training_unbalance_rate,
                    max_features = 5,
                    max_depth = 200
                ),
                DecisionTreeEdgePrediction(
                    edge_embedding_methods = edge_embedding_method,
                    use_scale_free_distribution = use_scale_free_distribution,
                    training_unbalance_rate = training_unbalance_rate,
                    max_features = 10,
                    max_depth = 200
                ),
                DecisionTreeEdgePrediction(
                    edge_embedding_methods = edge_embedding_method,
                    use_scale_free_distribution = use_scale_free_distribution,
                    training_unbalance_rate = training_unbalance_rate,
                    max_features = 10,
                    max_depth = 500
                ),
            ],
            node_features=embedding_bfs_10,
            smoke_test=False,
            enable_cache=True
        )
    ])
    results_10_tuning.to_csv(f'./RNA-KG_notebooks/Default_RNA-KG/reports/report7/csv/tuning/edge_prediction_fullrnakg_dim10_tree_tuning_{i}.csv')
    tree_10_tuning_results.append(results_10_tuning)

In [ ]:
from grape.edge_prediction import DecisionTreeEdgePrediction
from grape.edge_prediction import edge_prediction_evaluation

evaluation_schema = "Connected Monte Carlo"
edge_embedding_method = "Concatenate"
use_scale_free_distribution = True
training_unbalance_rate = 1/1
validation_unbalance_rate = 1/1

tree_10_tuning_results = []

for (i,train_graph) in enumerate(train_sets):
    results_10_tuning = pd.concat([
        edge_prediction_evaluation(
            graphs=train_graph,

            evaluation_schema = evaluation_schema,
            number_of_holdouts = 1,
            holdouts_kwargs=dict(train_size=0.7),
            use_scale_free_distribution=use_scale_free_distribution,
            validation_unbalance_rates=validation_unbalance_rate,
            
            models=[
                DecisionTreeEdgePrediction(
                    edge_embedding_methods = edge_embedding_method,
                    use_scale_free_distribution = use_scale_free_distribution,
                    training_unbalance_rate = training_unbalance_rate,
                    max_features = 'sqrt', #grape def
                    max_depth = 10, #grape def
                ),
                DecisionTreeEdgePrediction(
                    edge_embedding_methods = edge_embedding_method,
                    use_scale_free_distribution = use_scale_free_distribution,
                    training_unbalance_rate = training_unbalance_rate,
                    max_features = 'sqrt',
                    max_depth = 50,
                ),
                DecisionTreeEdgePrediction(
                    edge_embedding_methods = edge_embedding_method,
                    use_scale_free_distribution = use_scale_free_distribution,
                    training_unbalance_rate = training_unbalance_rate,
                    max_features = 5,
                    max_depth = 50,
                ),
                DecisionTreeEdgePrediction(
                    edge_embedding_methods = edge_embedding_method,
                    use_scale_free_distribution = use_scale_free_distribution,
                    training_unbalance_rate = training_unbalance_rate,
                    max_features = 10,
                    max_depth = 50,
                ),
                DecisionTreeEdgePrediction(
                    edge_embedding_methods = edge_embedding_method,
                    use_scale_free_distribution = use_scale_free_distribution,
                    training_unbalance_rate = training_unbalance_rate,
                    max_features='sqrt',
                    max_depth = 200
                ),
                DecisionTreeEdgePrediction(
                    edge_embedding_methods = edge_embedding_method,
                    use_scale_free_distribution = use_scale_free_distribution,
                    training_unbalance_rate = training_unbalance_rate,
                    max_features = 5,
                    max_depth = 200
                ),
                DecisionTreeEdgePrediction(
                    edge_embedding_methods = edge_embedding_method,
                    use_scale_free_distribution = use_scale_free_distribution,
                    training_unbalance_rate = training_unbalance_rate,
                    max_features = 10,
                    max_depth = 200
                ),
                DecisionTreeEdgePrediction(
                    edge_embedding_methods = edge_embedding_method,
                    use_scale_free_distribution = use_scale_free_distribution,
                    training_unbalance_rate = training_unbalance_rate,
                    max_features = 'sqrt',
                    max_depth = 500
                ),
                DecisionTreeEdgePrediction(
                    edge_embedding_methods = edge_embedding_method,
                    use_scale_free_distribution = use_scale_free_distribution,
                    training_unbalance_rate = training_unbalance_rate,
                    max_features = 5,
                    max_depth = 500
                ),
                DecisionTreeEdgePrediction(
                    edge_embedding_methods = edge_embedding_method,
                    use_scale_free_distribution = use_scale_free_distribution,
                    training_unbalance_rate = training_unbalance_rate,
                    max_features = 10,
                    max_depth = 500
                ),
            ],
            node_features=embedding_bfs_10,
            smoke_test=False,
            enable_cache=True
        )
    ])
    results_10_tuning.to_csv(f'./RNA-KG_notebooks/Default_RNA-KG/reports/report7/csv/tuning/edge_prediction_fullrnakg_dim10_tree_tuning_{i}.csv')
    tree_10_tuning_results.append(results_10_tuning)
full_df_10_tree_tuning = pd.concat(tree_10_tuning_results,ignore_index=True)
full_df_10_tree_tuning.to_csv('./RNA-KG_notebooks/Default_RNA-KG/reports/report7/csv/tuning/edge_prediction_fullrnakg_dim10_tree_tuning_combined.csv')

In [ ]:
full_df_10_tree_tuning[[('model_parameters', 'max_depth'),('model_parameters', 'max_features'),'evaluation_mode','balanced_accuracy']]\
    .groupby([('model_parameters', 'max_depth'),('model_parameters', 'max_features'),'evaluation_mode'])\
    .agg(['mean','std'])\
    .reset_index()\
    .sort_values([('balanced_accuracy','mean')],ascending=False)

In [ ]:
# # bar plot
# fig, ax = plt.subplots(figsize=(10, 6))
# barWidth = 0.4
# barsTrain = [
#     results_10[(results_10["evaluation_mode"]=="train") & (results_10["model_name"]=="Decision Tree Classifier")]["balanced_accuracy"].mean(),
#     results_10_tuning_5_50[(results_10_tuning_5_50["evaluation_mode"]=="train")]['balanced_accuracy'].mean(),
#     results_10_tuning_sqrt_200[(results_10_tuning_sqrt_200["evaluation_mode"]=="train")]['balanced_accuracy'].mean(),
#     results_10_tuning_5_200[(results_10_tuning_5_200["evaluation_mode"]=="train")]['balanced_accuracy'].mean(),
#     results_10_tuning_10_200[(results_10_tuning_10_200["evaluation_mode"]=="train")]['balanced_accuracy'].mean(),
#     results_10_tuning_10_500[(results_10_tuning_10_500["evaluation_mode"]=="train")]['balanced_accuracy'].mean(),
# ]
# barsTest = [
#     results_10[(results_10["evaluation_mode"]=="test") & (results_10["model_name"]=="Decision Tree Classifier")]["balanced_accuracy"].mean(),
#     results_10_tuning_5_50[(results_10_tuning_5_50["evaluation_mode"]=="test")]['balanced_accuracy'].mean(),
#     results_10_tuning_sqrt_200[(results_10_tuning_sqrt_200["evaluation_mode"]=="test")]['balanced_accuracy'].mean(),
#     results_10_tuning_5_200[(results_10_tuning_5_200["evaluation_mode"]=="test")]['balanced_accuracy'].mean(),
#     results_10_tuning_10_200[(results_10_tuning_10_200["evaluation_mode"]=="test")]['balanced_accuracy'].mean(),
#     results_10_tuning_10_500[(results_10_tuning_10_500["evaluation_mode"]=="test")]['balanced_accuracy'].mean(),
# ]
# r1 = range(len(barsTrain))
# r2 = [x + barWidth for x in r1]
# plt.bar(r1, barsTrain, width=barWidth, edgecolor='grey', label='Train')
# plt.bar(r2, barsTest, width=barWidth, edgecolor='grey', label='Test')
# plt.ylim(0,1)
# plt.xlabel('Model parameters: max features-max depth')
# plt.xticks([r + barWidth/2 for r in range(len(barsTrain))], ['sqrt-10','5-50','sqrt-200','5-200','10-200','10-500'])
# plt.ylabel('Balanced accuracy')
# # prevent legend from overlapping with the plot
# for i in range(len(barsTrain)):
#     plt.text(x=r1[i]-0.19, y=barsTrain[i]+0.01, s='{:.2%}'.format(barsTrain[i]), size=9)
#     plt.text(x=r2[i]-0.19, y=barsTest[i]+0.01, s='{:.2%}'.format(barsTest[i]), size=9)
# plt.legend(loc='upper left')
# plt.show()

In [ ]:
# results_10_tuning = pd.concat([
#     results_10_tuning_5_50,
#     results_10_tuning_sqrt_200,
#     results_10_tuning_5_200,
#     results_10_tuning_10_200,
#     results_10_tuning_10_500,
# ])

In [ ]:
# results_10_tuning.to_csv('./RNA-KG_notebooks/Default_RNA-KG/reports/report7/csv/edge_prediction_fullrnakg_dim10_decision_tree_tuning.csv')

## Tuned Decision Tree Edge Prediction

In [ ]:
from grape.edge_prediction import DecisionTreeEdgePrediction

evaluation_schema = "Connected Monte Carlo"
edge_embedding_method = "Concatenate"
use_scale_free_distribution = True
training_unbalance_rate = 1/1
validation_unbalance_rate = 1/1
random_state = 42

tree_10_tuned_results = []
for (i,train_graph) in enumerate(train_sets):
    print(f"Training on holdout {i}")
    model = DecisionTreeEdgePrediction(
        edge_embedding_methods = edge_embedding_method,
        use_scale_free_distribution = use_scale_free_distribution,
        training_unbalance_rate = training_unbalance_rate,
        max_features = 10,
        max_depth = 50,
    )
    print("Fitting model")
    model.fit(train_graph, node_features=embedding_bfs_10)
    
    print("Predicting on test set")
    print("Generating negative test graph")
    negative_test_graph = undirected_rnakg.sample_negative_graph(
      number_of_negative_samples=test_sets[i].get_number_of_directed_edges(),
      random_state=random_state,
      use_scale_free_distribution=use_scale_free_distribution,
    )
    print("Predicting on positive test set")
    pos_test_pred = model.predict_proba(
        graph=test_sets[i],
        node_features=embedding_bfs_10,
        return_predictions_dataframe=True
    )
    print("Predicting on negative test set")
    neg_test_pred = model.predict_proba(
        graph=negative_test_graph,
        node_features=embedding_bfs_10,
        return_predictions_dataframe=True
    )
    print("Calculating balanced accuracy")
    pos_score = balanced_accuracy_score([True for _ in range(len(pos_test_pred))], pos_test_pred['prediction'].apply(lambda x:x>0.5))
    neg_score = balanced_accuracy_score([False for _ in range(len(neg_test_pred))], neg_test_pred['prediction'].apply(lambda x:x>0.5))

    print(f"Balanced accuracy positive score: {pos_score}")
    print(f"Balanced accuracy negative score: {neg_score}")
    avg_score = (pos_score+neg_score)/2
    print(f"Balanced accuracy mean score: {avg_score}")

    results_10_tuned = pd.DataFrame({
        "model_name": [model.__class__.__name__],

        "pos_balanced_accuracy": [pos_score],
        "neg_balanced_accuracy": [neg_score],
        "overall_balanced_accuracy": [avg_score]
    })

    tree_10_tuned_results.append(results_10_tuned)
full_df_10_tree_tuned = pd.concat(tree_10_tuned_results,ignore_index=True)
full_df_10_tree_tuned.to_csv('./RNA-KG_notebooks/Default_RNA-KG/reports/report7/csv/edge_prediction_fullrnakg_dim10_tree_tuned.csv')

In [ ]:
full_df_10_tree_tuned['overall_balanced_accuracy'].mean(),full_df_10_tree_tuned['overall_balanced_accuracy'].std()

## Parameter Tuning on Random Forest

In [ ]:
from grape.edge_prediction import RandomForestEdgePrediction
from grape.edge_prediction import edge_prediction_evaluation

evaluation_schema = "Connected Monte Carlo"
edge_embedding_method = "Concatenate"
use_scale_free_distribution = True
training_unbalance_rate = 1/1
validation_unbalance_rate = 1/1

forest_10_tuning_results = []

for (i,train_graph) in enumerate(train_sets):
    results_10_tuning = pd.concat([
        edge_prediction_evaluation(
            graphs=train_graph,

            evaluation_schema = evaluation_schema,
            number_of_holdouts = 5,
            holdouts_kwargs=dict(train_size=0.7),
            use_scale_free_distribution=use_scale_free_distribution,
            validation_unbalance_rates=validation_unbalance_rate,
            
            models=[
                RandomForestEdgePrediction(
                    edge_embedding_methods = edge_embedding_method,
                    use_scale_free_distribution = use_scale_free_distribution,
                    training_unbalance_rate = training_unbalance_rate,
                    max_features = 10, 
                    max_depth = 50,
                    n_estimators= 10,
                    n_jobs= -1
                ),
                RandomForestEdgePrediction(
                    edge_embedding_methods = edge_embedding_method,
                    use_scale_free_distribution = use_scale_free_distribution,
                    training_unbalance_rate = training_unbalance_rate,
                    max_features = 10, 
                    max_depth = 50,
                    n_estimators= 50,
                    n_jobs= -1
                ),
                RandomForestEdgePrediction(
                    edge_embedding_methods = edge_embedding_method,
                    use_scale_free_distribution = use_scale_free_distribution,
                    training_unbalance_rate = training_unbalance_rate,
                    max_features = 10, 
                    max_depth = 50,
                    n_estimators= 100,
                    n_jobs= -1
                ),
                RandomForestEdgePrediction(
                    edge_embedding_methods = edge_embedding_method,
                    use_scale_free_distribution = use_scale_free_distribution,
                    training_unbalance_rate = training_unbalance_rate,
                    max_features = 10, 
                    max_depth = 50,
                    n_estimators= 200,
                    n_jobs= -1
                ),
                 RandomForestEdgePrediction(
                    edge_embedding_methods = edge_embedding_method,
                    use_scale_free_distribution = use_scale_free_distribution,
                    training_unbalance_rate = training_unbalance_rate,
                    max_features = 10, 
                    max_depth = 50,
                    n_estimators= 500,
                    n_jobs= -1
                ),
            ],
            node_features=embedding_bfs_10,
            smoke_test=False,
            enable_cache=True
        )
    ])
    results_10_tuning.to_csv(f'./RNA-KG_notebooks/Default_RNA-KG/reports/report7/csv/tuning/edge_prediction_fullrnakg_dim10_forest_tuning_{i}.csv')
    forest_10_tuning_results.append(results_10_tuning)
full_df_10_forest_tuning = pd.concat(forest_10_tuning_results,ignore_index=True)
full_df_10_forest_tuning.to_csv('./RNA-KG_notebooks/Default_RNA-KG/reports/report7/csv/tuning/edge_prediction_fullrnakg_dim10_forest_tuning_combined.csv')

In [ ]:
full_df_10_forest_tuning[[('model_parameters', 'max_depth'),('model_parameters', 'max_features'),('model_parameters', 'n_estimators'),'evaluation_mode','balanced_accuracy']]\
    .groupby([('model_parameters', 'max_depth'),('model_parameters', 'max_features'),('model_parameters', 'n_estimators'),'evaluation_mode'])\
    .agg(['mean','std'])\
    .reset_index()\
    .sort_values([('balanced_accuracy','mean')],ascending=False)

## Tuned Random Forest Edge Prediction

In [ ]:
from grape.edge_prediction import RandomForestEdgePrediction

evaluation_schema = "Connected Monte Carlo"
edge_embedding_method = "Concatenate"
use_scale_free_distribution = True
training_unbalance_rate = 1/1
validation_unbalance_rate = 1/1
random_state = 42

forest_10_tuned_results = []
for (i,train_graph) in enumerate(train_sets):
    print(f"Training on holdout {i}")
    model = RandomForestEdgePrediction(
        edge_embedding_methods = edge_embedding_method,
        use_scale_free_distribution = use_scale_free_distribution,
        training_unbalance_rate = training_unbalance_rate,
        max_features = 10, 
        max_depth = 50,
        n_estimators= 200,
        n_jobs= -1
    )
    print("Fitting model")
    model.fit(train_graph, node_features=embedding_bfs_10)
    
    print("Predicting on test set")
    print("Generating negative test graph")
    negative_test_graph = undirected_rnakg.sample_negative_graph(
      number_of_negative_samples=test_sets[i].get_number_of_directed_edges(),
      random_state=random_state,
      use_scale_free_distribution=use_scale_free_distribution,
    )
    print("Predicting on positive test set")
    pos_test_pred = model.predict_proba(
        graph=test_sets[i],
        node_features=embedding_bfs_10,
        return_predictions_dataframe=True
    )
    print("Predicting on negative test set")
    neg_test_pred = model.predict_proba(
        graph=negative_test_graph,
        node_features=embedding_bfs_10,
        return_predictions_dataframe=True
    )
    print("Calculating balanced accuracy")
    pos_score = balanced_accuracy_score([True for _ in range(len(pos_test_pred))], pos_test_pred['prediction'].apply(lambda x:x>0.5))
    neg_score = balanced_accuracy_score([False for _ in range(len(neg_test_pred))], neg_test_pred['prediction'].apply(lambda x:x>0.5))

    print(f"Balanced accuracy positive score: {pos_score}")
    print(f"Balanced accuracy negative score: {neg_score}")
    avg_score = (pos_score+neg_score)/2
    print(f"Balanced accuracy mean score: {avg_score}")

    results_10_tuned = pd.DataFrame({
        "model_name": [model.__class__.__name__],

        "pos_balanced_accuracy": [pos_score],
        "neg_balanced_accuracy": [neg_score],
        "overall_balanced_accuracy": [avg_score]
    })

    forest_10_tuned_results.append(results_10_tuned)
full_df_10_forest_tuned = pd.concat(forest_10_tuned_results,ignore_index=True)
full_df_10_forest_tuned.to_csv('./RNA-KG_notebooks/Default_RNA-KG/reports/report7/csv/edge_prediction_fullrnakg_dim10_forest_tuned.csv')

In [ ]:
full_df_10_forest_tuned['overall_balanced_accuracy'].mean(),full_df_10_forest_tuned['overall_balanced_accuracy'].std()

# 50D embedding

In [ ]:
# load embedding
embedding_bfs_50 = cache.load_embeddings('Node2VecSkipGramEnsmallen_BFS_50_fixed')
print(type(embedding_bfs_50))

## Parameter Tuning on Decision Tree

In [ ]:
from grape.edge_prediction import DecisionTreeEdgePrediction
from grape.edge_prediction import edge_prediction_evaluation

evaluation_schema = "Connected Monte Carlo"
edge_embedding_method = "Concatenate"
use_scale_free_distribution = True
training_unbalance_rate = 1/1
validation_unbalance_rate = 1/1

tree_50_tuning_results = []

for (i,train_graph) in enumerate(train_sets):
    results_50_tuning = pd.concat([
        edge_prediction_evaluation(
            graphs=train_graph,

            evaluation_schema = evaluation_schema,
            number_of_holdouts = 1,
            holdouts_kwargs=dict(train_size=0.7),
            use_scale_free_distribution=use_scale_free_distribution,
            validation_unbalance_rates=validation_unbalance_rate,
            
            models=[
                DecisionTreeEdgePrediction(
                    edge_embedding_methods = edge_embedding_method,
                    use_scale_free_distribution = use_scale_free_distribution,
                    training_unbalance_rate = training_unbalance_rate,
                    max_features = 'sqrt', #grape def
                    max_depth = 10, #grape def
                ),
                DecisionTreeEdgePrediction(
                    edge_embedding_methods = edge_embedding_method,
                    use_scale_free_distribution = use_scale_free_distribution,
                    training_unbalance_rate = training_unbalance_rate,
                    max_features = 'sqrt',
                    max_depth = 50,
                ),
                DecisionTreeEdgePrediction(
                    edge_embedding_methods = edge_embedding_method,
                    use_scale_free_distribution = use_scale_free_distribution,
                    training_unbalance_rate = training_unbalance_rate,
                    max_features = 25,
                    max_depth = 50,
                ),
                DecisionTreeEdgePrediction(
                    edge_embedding_methods = edge_embedding_method,
                    use_scale_free_distribution = use_scale_free_distribution,
                    training_unbalance_rate = training_unbalance_rate,
                    max_features = 50,
                    max_depth = 50,
                ),
                DecisionTreeEdgePrediction(
                    edge_embedding_methods = edge_embedding_method,
                    use_scale_free_distribution = use_scale_free_distribution,
                    training_unbalance_rate = training_unbalance_rate,
                    max_features= 'sqrt',
                    max_depth = 200
                ),
                DecisionTreeEdgePrediction(
                    edge_embedding_methods = edge_embedding_method,
                    use_scale_free_distribution = use_scale_free_distribution,
                    training_unbalance_rate = training_unbalance_rate,
                    max_features = 25,
                    max_depth = 200
                ),
                DecisionTreeEdgePrediction(
                    edge_embedding_methods = edge_embedding_method,
                    use_scale_free_distribution = use_scale_free_distribution,
                    training_unbalance_rate = training_unbalance_rate,
                    max_features = 50,
                    max_depth = 200
                ),
                DecisionTreeEdgePrediction(
                    edge_embedding_methods = edge_embedding_method,
                    use_scale_free_distribution = use_scale_free_distribution,
                    training_unbalance_rate = training_unbalance_rate,
                    max_features = 'sqrt',
                    max_depth = 500
                ),
                DecisionTreeEdgePrediction(
                    edge_embedding_methods = edge_embedding_method,
                    use_scale_free_distribution = use_scale_free_distribution,
                    training_unbalance_rate = training_unbalance_rate,
                    max_features = 25,
                    max_depth = 500
                ),
                DecisionTreeEdgePrediction(
                    edge_embedding_methods = edge_embedding_method,
                    use_scale_free_distribution = use_scale_free_distribution,
                    training_unbalance_rate = training_unbalance_rate,
                    max_features = 50,
                    max_depth = 500
                ),
            ],
            node_features=embedding_bfs_50,
            smoke_test=False,
            enable_cache=True
        )
    ])
    results_50_tuning.to_csv(f'./RNA-KG_notebooks/Default_RNA-KG/reports/report7/csv/tuning/edge_prediction_fullrnakg_dim50_tree_tuning_{i}.csv')
    tree_50_tuning_results.append(results_50_tuning)
full_df_50_tree_tuning = pd.concat(tree_50_tuning_results,ignore_index=True)
full_df_50_tree_tuning.to_csv('./RNA-KG_notebooks/Default_RNA-KG/reports/report7/csv/tuning/edge_prediction_fullrnakg_dim50_tree_tuning_combined.csv')

In [ ]:
full_df_50_tree_tuning[[('model_parameters', 'max_depth'),('model_parameters', 'max_features'),'evaluation_mode','balanced_accuracy','time_required_for_training','time_required_for_evaluation']]\
    .groupby([('model_parameters', 'max_depth'),('model_parameters', 'max_features'),'evaluation_mode'])\
    .agg(['mean','std'])\
    .reset_index()\
    .sort_values([('balanced_accuracy','mean')],ascending=False)

## Tuned Decision Tree Edge Prediction

In [ ]:
from grape.edge_prediction import DecisionTreeEdgePrediction
import time

evaluation_schema = "Connected Monte Carlo"
edge_embedding_method = "Concatenate"
use_scale_free_distribution = True
training_unbalance_rate = 1/1
validation_unbalance_rate = 1/1
random_state = 42

tree_50_tuned_results = []
for (i,train_graph) in enumerate(train_sets):
    start_training_time = time.time()
    print(f"Training on holdout {i}")
    model = DecisionTreeEdgePrediction(
        edge_embedding_methods = edge_embedding_method,
        use_scale_free_distribution = use_scale_free_distribution,
        training_unbalance_rate = training_unbalance_rate,
        max_features = 50,
        max_depth = 500,
    )
    print("Fitting model")
    model.fit(train_graph, node_features=embedding_bfs_50)
    
    training_time = time.time() - start_training_time
    print(f"Training time: {training_time:.2f}s")


    testing_time_start = time.time()

    print("Predicting on test set")
    print("Generating negative test graph")
    negative_test_graph = undirected_rnakg.sample_negative_graph(
      number_of_negative_samples=test_sets[i].get_number_of_directed_edges(),
      random_state=random_state,
      use_scale_free_distribution=use_scale_free_distribution,
    )
    print("Predicting on positive test set")
    pos_test_pred = model.predict_proba(
        graph=test_sets[i],
        node_features=embedding_bfs_50,
        return_predictions_dataframe=True
    )
    print("Predicting on negative test set")
    neg_test_pred = model.predict_proba(
        graph=negative_test_graph,
        node_features=embedding_bfs_50,
        return_predictions_dataframe=True
    )

    testing_time = time.time() - testing_time_start
    print(f"Testing time: {testing_time:.2f}s")

    print("Calculating balanced accuracy")
    pos_score = balanced_accuracy_score([True for _ in range(len(pos_test_pred))], pos_test_pred['prediction'].apply(lambda x:x>0.5))
    neg_score = balanced_accuracy_score([False for _ in range(len(neg_test_pred))], neg_test_pred['prediction'].apply(lambda x:x>0.5))

    print(f"Balanced accuracy positive score: {pos_score}")
    print(f"Balanced accuracy negative score: {neg_score}")
    avg_score = (pos_score+neg_score)/2
    print(f"Balanced accuracy mean score: {avg_score}")

    results_50_tuned = pd.DataFrame({
        "model_name": [model.__class__.__name__],

        "pos_balanced_accuracy": [pos_score],
        "neg_balanced_accuracy": [neg_score],
        "overall_balanced_accuracy": [avg_score],
        "time_required_for_training": [training_time],
        "time_required_for_evaluation": [testing_time]
    })

    tree_50_tuned_results.append(results_50_tuned)
full_df_50_tree_tuned = pd.concat(tree_50_tuned_results,ignore_index=True)
full_df_50_tree_tuned.to_csv('./RNA-KG_notebooks/Default_RNA-KG/reports/report7/csv/edge_prediction_fullrnakg_dim50_tree_tuned.csv')

In [ ]:
print(f"{full_df_50_tree_tuned['overall_balanced_accuracy'].mean():.2%} ± {full_df_50_tree_tuned['overall_balanced_accuracy'].std():.2%}")

## Parameter Tuning on Random Forest

In [ ]:
from grape.edge_prediction import RandomForestEdgePrediction
from grape.edge_prediction import edge_prediction_evaluation

evaluation_schema = "Connected Monte Carlo"
edge_embedding_method = "Concatenate"
use_scale_free_distribution = True
training_unbalance_rate = 1/1
validation_unbalance_rate = 1/1

forest_50_tuning_results = []

for (i,train_graph) in enumerate(train_sets):
    results_50_tuning = pd.concat([
        edge_prediction_evaluation(
            graphs=train_graph,

            evaluation_schema = evaluation_schema,
            number_of_holdouts = 5,
            holdouts_kwargs=dict(train_size=0.7),
            use_scale_free_distribution=use_scale_free_distribution,
            validation_unbalance_rates=validation_unbalance_rate,
            
            models=[
                RandomForestEdgePrediction(
                    edge_embedding_methods = edge_embedding_method,
                    use_scale_free_distribution = use_scale_free_distribution,
                    training_unbalance_rate = training_unbalance_rate,
                    max_features = 50, 
                    max_depth = 200,
                    n_estimators= 10,
                    n_jobs= -1
                ),
                RandomForestEdgePrediction(
                    edge_embedding_methods = edge_embedding_method,
                    use_scale_free_distribution = use_scale_free_distribution,
                    training_unbalance_rate = training_unbalance_rate,
                    max_features = 50, 
                    max_depth = 200,
                    n_estimators= 50,
                    n_jobs= -1
                ),
                RandomForestEdgePrediction(
                    edge_embedding_methods = edge_embedding_method,
                    use_scale_free_distribution = use_scale_free_distribution,
                    training_unbalance_rate = training_unbalance_rate,
                    max_features = 50, 
                    max_depth = 200,
                    n_estimators= 100,
                    n_jobs= -1
                ),
                RandomForestEdgePrediction(
                    edge_embedding_methods = edge_embedding_method,
                    use_scale_free_distribution = use_scale_free_distribution,
                    training_unbalance_rate = training_unbalance_rate,
                    max_features = 50, 
                    max_depth = 200,
                    n_estimators= 200,
                    n_jobs= -1
                ),
                 RandomForestEdgePrediction(
                    edge_embedding_methods = edge_embedding_method,
                    use_scale_free_distribution = use_scale_free_distribution,
                    training_unbalance_rate = training_unbalance_rate,
                    max_features = 50, 
                    max_depth = 200,
                    n_estimators= 500,
                    n_jobs= -1
                ),
            ],
            node_features=embedding_bfs_50,
            smoke_test=False,
            enable_cache=True
        )
    ])
    results_50_tuning.to_csv(f'./RNA-KG_notebooks/Default_RNA-KG/reports/report7/csv/tuning/edge_prediction_fullrnakg_dim50_forest_tuning_{i}.csv')
    forest_50_tuning_results.append(results_50_tuning)
full_df_50_forest_tuning = pd.concat(forest_50_tuning_results,ignore_index=True)
full_df_50_forest_tuning.to_csv('./RNA-KG_notebooks/Default_RNA-KG/reports/report7/csv/tuning/edge_prediction_fullrnakg_dim50_forest_tuning_combined.csv')

In [ ]:
full_df_50_forest_tuning[[('model_parameters', 'max_depth'),('model_parameters', 'max_features'),('model_parameters', 'n_estimators'),'evaluation_mode','balanced_accuracy','time_required_for_training','time_required_for_evaluation']]\
    .groupby([('model_parameters', 'max_depth'),('model_parameters', 'max_features'),('model_parameters', 'n_estimators'),'evaluation_mode'])\
    .agg(['mean','std'])\
    .sort_values([('balanced_accuracy','mean')],ascending=False)\
    .reset_index()

In [ ]:
(full_df_50_forest_tuning['time_required_for_training'] + full_df_50_forest_tuning['time_required_for_evaluation']).sum()/2

## Tuned Random Forest Edge Prediction

In [ ]:
from grape.edge_prediction import RandomForestEdgePrediction

evaluation_schema = "Connected Monte Carlo"
edge_embedding_method = "Concatenate"
use_scale_free_distribution = True
training_unbalance_rate = 1/1
validation_unbalance_rate = 1/1
random_state = 42

forest_50_tuned_results = []
for (i,train_graph) in enumerate(train_sets):
    print(f"Training on holdout {i}")
    model = RandomForestEdgePrediction(
        edge_embedding_methods = edge_embedding_method,
        use_scale_free_distribution = use_scale_free_distribution,
        training_unbalance_rate = training_unbalance_rate,
        max_features = 50, 
        max_depth = 200,
        n_estimators= 500,
        n_jobs= -1
    )
    print("Fitting model")
    model.fit(train_graph, node_features=embedding_bfs_50)
    
    print("Predicting on test set")
    print("Generating negative test graph")
    negative_test_graph = undirected_rnakg.sample_negative_graph(
      number_of_negative_samples=test_sets[i].get_number_of_directed_edges(),
      random_state=random_state,
      use_scale_free_distribution=use_scale_free_distribution,
    )
    print("Predicting on positive test set")
    pos_test_pred = model.predict_proba(
        graph=test_sets[i],
        node_features=embedding_bfs_50,
        return_predictions_dataframe=True
    )
    print("Predicting on negative test set")
    neg_test_pred = model.predict_proba(
        graph=negative_test_graph,
        node_features=embedding_bfs_50,
        return_predictions_dataframe=True
    )
    print("Calculating balanced accuracy")
    pos_score = balanced_accuracy_score([True for _ in range(len(pos_test_pred))], pos_test_pred['prediction'].apply(lambda x:x>0.5))
    neg_score = balanced_accuracy_score([False for _ in range(len(neg_test_pred))], neg_test_pred['prediction'].apply(lambda x:x>0.5))

    print(f"Balanced accuracy positive score: {pos_score}")
    print(f"Balanced accuracy negative score: {neg_score}")
    avg_score = (pos_score+neg_score)/2
    print(f"Balanced accuracy mean score: {avg_score}")

    results_50_tuned = pd.DataFrame({
        "model_name": [model.__class__.__name__],

        "pos_balanced_accuracy": [pos_score],
        "neg_balanced_accuracy": [neg_score],
        "overall_balanced_accuracy": [avg_score]
    })

    forest_50_tuned_results.append(results_50_tuned)
full_df_50_forest_tuned = pd.concat(forest_50_tuned_results,ignore_index=True)
full_df_50_forest_tuned.to_csv('./RNA-KG_notebooks/Default_RNA-KG/reports/report7/csv/edge_prediction_fullrnakg_dim50_forest_tuned.csv')

In [ ]:
full_df_50_forest_tuned['overall_balanced_accuracy'].mean(),full_df_50_forest_tuned['overall_balanced_accuracy'].std()

# 100D embedding

In [ ]:
# load embedding
embedding_bfs_100 = cache.load_embeddings('Node2VecSkipGramEnsmallen_BFS_100_fixed')
print(type(embedding_bfs_100))

## Parameter Tuning on Decision Tree

In [ ]:
from grape.edge_prediction import DecisionTreeEdgePrediction
from grape.edge_prediction import edge_prediction_evaluation

evaluation_schema = "Connected Monte Carlo"
edge_embedding_method = "Concatenate"
use_scale_free_distribution = True
training_unbalance_rate = 1/1
validation_unbalance_rate = 1/1

tree_100_tuning_results = []

for (i,train_graph) in enumerate(train_sets):
    results_100_tuning = pd.concat([
        edge_prediction_evaluation(
            graphs=train_graph,

            evaluation_schema = evaluation_schema,
            number_of_holdouts = 1,
            holdouts_kwargs=dict(train_size=0.7),
            use_scale_free_distribution=use_scale_free_distribution,
            validation_unbalance_rates=validation_unbalance_rate,
            
            models=[
                DecisionTreeEdgePrediction(
                    edge_embedding_methods = edge_embedding_method,
                    use_scale_free_distribution = use_scale_free_distribution,
                    training_unbalance_rate = training_unbalance_rate,
                    max_features = 'sqrt', #grape def
                    max_depth = 10, #grape def
                ),
                DecisionTreeEdgePrediction(
                    edge_embedding_methods = edge_embedding_method,
                    use_scale_free_distribution = use_scale_free_distribution,
                    training_unbalance_rate = training_unbalance_rate,
                    max_features = 'sqrt',
                    max_depth = 50,
                ),
                DecisionTreeEdgePrediction(
                    edge_embedding_methods = edge_embedding_method,
                    use_scale_free_distribution = use_scale_free_distribution,
                    training_unbalance_rate = training_unbalance_rate,
                    max_features = 50,
                    max_depth = 50,
                ),
                DecisionTreeEdgePrediction(
                    edge_embedding_methods = edge_embedding_method,
                    use_scale_free_distribution = use_scale_free_distribution,
                    training_unbalance_rate = training_unbalance_rate,
                    max_features = 100,
                    max_depth = 50,
                ),
                DecisionTreeEdgePrediction(
                    edge_embedding_methods = edge_embedding_method,
                    use_scale_free_distribution = use_scale_free_distribution,
                    training_unbalance_rate = training_unbalance_rate,
                    max_features= 'sqrt',
                    max_depth = 200
                ),
                DecisionTreeEdgePrediction(
                    edge_embedding_methods = edge_embedding_method,
                    use_scale_free_distribution = use_scale_free_distribution,
                    training_unbalance_rate = training_unbalance_rate,
                    max_features = 50,
                    max_depth = 200
                ),
                DecisionTreeEdgePrediction(
                    edge_embedding_methods = edge_embedding_method,
                    use_scale_free_distribution = use_scale_free_distribution,
                    training_unbalance_rate = training_unbalance_rate,
                    max_features = 100,
                    max_depth = 200
                ),
                DecisionTreeEdgePrediction(
                    edge_embedding_methods = edge_embedding_method,
                    use_scale_free_distribution = use_scale_free_distribution,
                    training_unbalance_rate = training_unbalance_rate,
                    max_features = 'sqrt',
                    max_depth = 500
                ),
                DecisionTreeEdgePrediction(
                    edge_embedding_methods = edge_embedding_method,
                    use_scale_free_distribution = use_scale_free_distribution,
                    training_unbalance_rate = training_unbalance_rate,
                    max_features = 50,
                    max_depth = 500
                ),
                DecisionTreeEdgePrediction(
                    edge_embedding_methods = edge_embedding_method,
                    use_scale_free_distribution = use_scale_free_distribution,
                    training_unbalance_rate = training_unbalance_rate,
                    max_features = 100,
                    max_depth = 500
                ),
            ],
            node_features=embedding_bfs_100,
            smoke_test=False,
            enable_cache=True
        )
    ])
    results_100_tuning.to_csv(f'./RNA-KG_notebooks/Default_RNA-KG/reports/report7/csv/tuning/edge_prediction_fullrnakg_dim100_tree_tuning_{i}.csv')
    tree_100_tuning_results.append(results_100_tuning)
full_df_100_tree_tuning = pd.concat(tree_100_tuning_results,ignore_index=True)
full_df_100_tree_tuning.to_csv('./RNA-KG_notebooks/Default_RNA-KG/reports/report7/csv/tuning/edge_prediction_fullrnakg_dim100_tree_tuning_combined.csv')

In [ ]:
full_df_100_tree_tuning[[('model_parameters', 'max_depth'),('model_parameters', 'max_features'),'evaluation_mode','balanced_accuracy','time_required_for_training','time_required_for_evaluation']]\
    .groupby([('model_parameters', 'max_depth'),('model_parameters', 'max_features'),'evaluation_mode'])\
    .agg(['mean','std'])\
    .reset_index()\
    .sort_values([('balanced_accuracy','mean')],ascending=False)

## Tuned Decision Tree Edge Prediction

In [ ]:
from grape.edge_prediction import DecisionTreeEdgePrediction
import time 

evaluation_schema = "Connected Monte Carlo"
edge_embedding_method = "Concatenate"
use_scale_free_distribution = True
training_unbalance_rate = 1/1
validation_unbalance_rate = 1/1
random_state = 42

tree_100_tuned_results = []
for (i,train_graph) in enumerate(train_sets):
    print(f"Training on holdout {i}")

    training_time_start = time.time()

    model = DecisionTreeEdgePrediction(
        edge_embedding_methods = edge_embedding_method,
        use_scale_free_distribution = use_scale_free_distribution,
        training_unbalance_rate = training_unbalance_rate,
        max_features = 50,
        max_depth = 50,
    )
    print("Fitting model")
    model.fit(train_graph, node_features=embedding_bfs_100)

    training_time = time.time() - training_time_start
    print(f"Training time: {training_time:.2f}s")

    testing_time_start = time.time()
    
    print("Predicting on test set")
    print("Generating negative test graph")
    negative_test_graph = undirected_rnakg.sample_negative_graph(
      number_of_negative_samples=test_sets[i].get_number_of_directed_edges(),
      random_state=random_state,
      use_scale_free_distribution=use_scale_free_distribution,
    )
    print("Predicting on positive test set")
    pos_test_pred = model.predict_proba(
        graph=test_sets[i],
        node_features=embedding_bfs_100,
        return_predictions_dataframe=True
    )
    print("Predicting on negative test set")
    neg_test_pred = model.predict_proba(
        graph=negative_test_graph,
        node_features=embedding_bfs_100,
        return_predictions_dataframe=True
    )

    testing_time = time.time() - testing_time_start
    print(f"Testing time: {testing_time:.2f}s")

    print("Calculating balanced accuracy")
    pos_score = balanced_accuracy_score([True for _ in range(len(pos_test_pred))], pos_test_pred['prediction'].apply(lambda x:x>0.5))
    neg_score = balanced_accuracy_score([False for _ in range(len(neg_test_pred))], neg_test_pred['prediction'].apply(lambda x:x>0.5))

    print(f"Balanced accuracy positive score: {pos_score}")
    print(f"Balanced accuracy negative score: {neg_score}")
    avg_score = (pos_score+neg_score)/2
    print(f"Balanced accuracy mean score: {avg_score}")

    results_100_tuned = pd.DataFrame({
        "model_name": [model.__class__.__name__],

        "pos_balanced_accuracy": [pos_score],
        "neg_balanced_accuracy": [neg_score],
        "overall_balanced_accuracy": [avg_score],
        "time_required_for_training": [training_time],
        "time_required_for_evaluation": [testing_time]
    })

    tree_100_tuned_results.append(results_100_tuned)
full_df_100_tree_tuned = pd.concat(tree_100_tuned_results,ignore_index=True)
full_df_100_tree_tuned.to_csv('./RNA-KG_notebooks/Default_RNA-KG/reports/report7/csv/edge_prediction_fullrnakg_dim100_tree_tuned.csv')

In [ ]:
full_df_100_tree_tuned['overall_balanced_accuracy'].mean(),full_df_100_tree_tuned['overall_balanced_accuracy'].std()

In [ ]:
print(f"{full_df_100_tree_tuned['overall_balanced_accuracy'].mean():.2%} ± {full_df_100_tree_tuned['overall_balanced_accuracy'].std():.2%}")

## Parameter Tuning on Random Forest

In [ ]:
from grape.edge_prediction import RandomForestEdgePrediction
from grape.edge_prediction import edge_prediction_evaluation

evaluation_schema = "Connected Monte Carlo"
edge_embedding_method = "Concatenate"
use_scale_free_distribution = True
training_unbalance_rate = 1/1
validation_unbalance_rate = 1/1

forest_100_tuning_results = []

for (i,train_graph) in enumerate(train_sets):
    results_100_tuning = pd.concat([
        edge_prediction_evaluation(
            graphs=train_graph,

            evaluation_schema = evaluation_schema,
            number_of_holdouts = 1,
            holdouts_kwargs=dict(train_size=0.7),
            use_scale_free_distribution=use_scale_free_distribution,
            validation_unbalance_rates=validation_unbalance_rate,
            
            models=[
                RandomForestEdgePrediction(
                    edge_embedding_methods = edge_embedding_method,
                    use_scale_free_distribution = use_scale_free_distribution,
                    training_unbalance_rate = training_unbalance_rate,
                    max_features = 50, 
                    max_depth = 50,
                    n_estimators= 10,
                    n_jobs= -1
                ),
                RandomForestEdgePrediction(
                    edge_embedding_methods = edge_embedding_method,
                    use_scale_free_distribution = use_scale_free_distribution,
                    training_unbalance_rate = training_unbalance_rate,
                    max_features = 50, 
                    max_depth = 50,
                    n_estimators= 50,
                    n_jobs= -1
                ),
                RandomForestEdgePrediction(
                    edge_embedding_methods = edge_embedding_method,
                    use_scale_free_distribution = use_scale_free_distribution,
                    training_unbalance_rate = training_unbalance_rate,
                    max_features = 50, 
                    max_depth = 50,
                    n_estimators= 100,
                    n_jobs= -1
                ),
                RandomForestEdgePrediction(
                    edge_embedding_methods = edge_embedding_method,
                    use_scale_free_distribution = use_scale_free_distribution,
                    training_unbalance_rate = training_unbalance_rate,
                    max_features = 50, 
                    max_depth = 50,
                    n_estimators= 200,
                    n_jobs= -1
                ),
                 RandomForestEdgePrediction(
                    edge_embedding_methods = edge_embedding_method,
                    use_scale_free_distribution = use_scale_free_distribution,
                    training_unbalance_rate = training_unbalance_rate,
                    max_features = 50, 
                    max_depth = 50,
                    n_estimators= 500,
                    n_jobs= -1
                ),
            ],
            node_features=embedding_bfs_100,
            smoke_test=False,
            enable_cache=True
        )
    ])
    results_100_tuning.to_csv(f'./RNA-KG_notebooks/Default_RNA-KG/reports/report7/csv/tuning/edge_prediction_fullrnakg_dim100_forest_tuning_{i}.csv')
    forest_100_tuning_results.append(results_100_tuning)
full_df_100_forest_tuning = pd.concat(forest_100_tuning_results,ignore_index=True)
full_df_100_forest_tuning.to_csv('./RNA-KG_notebooks/Default_RNA-KG/reports/report7/csv/tuning/edge_prediction_fullrnakg_dim100_forest_tuning_combined.csv')

In [ ]:
full_df_100_forest_tuning[[('model_parameters', 'max_depth'),('model_parameters', 'max_features'),('model_parameters', 'n_estimators'),'evaluation_mode','balanced_accuracy','time_required_for_training','time_required_for_evaluation']]\
    .groupby([('model_parameters', 'max_depth'),('model_parameters', 'max_features'),('model_parameters', 'n_estimators'),'evaluation_mode'])\
    .agg(['mean','std'])\
    .sort_values([('balanced_accuracy','mean')],ascending=False)\
    .reset_index()

## Tuned Random Forest Edge Prediction

In [ ]:
from grape.edge_prediction import RandomForestEdgePrediction

evaluation_schema = "Connected Monte Carlo"
edge_embedding_method = "Concatenate"
use_scale_free_distribution = True
training_unbalance_rate = 1/1
validation_unbalance_rate = 1/1
random_state = 42

forest_100_tuned_results = []
for (i,train_graph) in enumerate(train_sets):
    print(f"Training on holdout {i}")
    model = RandomForestEdgePrediction(
        edge_embedding_methods = edge_embedding_method,
        use_scale_free_distribution = use_scale_free_distribution,
        training_unbalance_rate = training_unbalance_rate,
        max_features = 50, 
        max_depth = 50,
        n_estimators= 500,
        n_jobs= -1
    )
    print("Fitting model")
    model.fit(train_graph, node_features=embedding_bfs_100)
    
    print("Predicting on test set")
    print("Generating negative test graph")
    negative_test_graph = undirected_rnakg.sample_negative_graph(
      number_of_negative_samples=test_sets[i].get_number_of_directed_edges(),
      random_state=random_state,
      use_scale_free_distribution=use_scale_free_distribution,
    )
    print("Predicting on positive test set")
    pos_test_pred = model.predict_proba(
        graph=test_sets[i],
        node_features=embedding_bfs_100,
        return_predictions_dataframe=True
    )
    print("Predicting on negative test set")
    neg_test_pred = model.predict_proba(
        graph=negative_test_graph,
        node_features=embedding_bfs_100,
        return_predictions_dataframe=True
    )
    print("Calculating balanced accuracy")
    pos_score = balanced_accuracy_score([True for _ in range(len(pos_test_pred))], pos_test_pred['prediction'].apply(lambda x:x>0.5))
    neg_score = balanced_accuracy_score([False for _ in range(len(neg_test_pred))], neg_test_pred['prediction'].apply(lambda x:x>0.5))

    print(f"Balanced accuracy positive score: {pos_score}")
    print(f"Balanced accuracy negative score: {neg_score}")
    avg_score = (pos_score+neg_score)/2
    print(f"Balanced accuracy mean score: {avg_score}")

    results_100_tuned = pd.DataFrame({
        "model_name": [model.__class__.__name__],

        "pos_balanced_accuracy": [pos_score],
        "neg_balanced_accuracy": [neg_score],
        "overall_balanced_accuracy": [avg_score]
    })

    forest_100_tuned_results.append(results_100_tuned)
full_df_100_forest_tuned = pd.concat(forest_100_tuned_results,ignore_index=True)
full_df_100_forest_tuned.to_csv('./RNA-KG_notebooks/Default_RNA-KG/reports/report7/csv/edge_prediction_fullrnakg_dim100_forest_tuned.csv')

In [ ]:
full_df_100_forest_tuned = pd.concat(forest_100_tuned_results,ignore_index=True)
full_df_100_forest_tuned

In [ ]:
full_df_100_forest_tuned['overall_balanced_accuracy'].mean(),full_df_100_forest_tuned['overall_balanced_accuracy'].std()

In [ ]:
print(f"{full_df_100_forest_tuned['overall_balanced_accuracy'].mean():.2%} ± {full_df_100_forest_tuned['overall_balanced_accuracy'].std():.2%}")